# Importar librerias

In [4]:
# Importar librerías
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# Cargar datos


In [6]:
# Fijar semillas para reproducibilidad simple
def fijar_semillas(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
fijar_semillas(42)

# Definir rutas de los archivos 
RUTA_MOVIES  = "./movies.csv"
RUTA_RATINGS = "./ratings.csv"

# Cargar CSVs con pandas (tipos básicos para ahorrar memoria)
dtype_movies = {"movieId": "int64", "title": "string", "genres": "string"}
dtype_ratings = {"userId": "int64", "movieId": "int64", "rating": "float32"}

# Leer datos
movies  = pd.read_csv(RUTA_MOVIES,  dtype=dtype_movies)
ratings = pd.read_csv(RUTA_RATINGS, dtype=dtype_ratings)

# Comprobar columnas mínimas
cols_movies_ok  = {"movieId", "title"}.issubset(movies.columns)
cols_ratings_ok = {"userId", "movieId", "rating"}.issubset(ratings.columns)
assert cols_movies_ok and cols_ratings_ok, "Faltan columnas requeridas en los CSVs."

# - Eliminar filas con NaN en columnas clave
movies  = movies.dropna(subset=["movieId", "title"]).reset_index(drop=True)
ratings = ratings.dropna(subset=["userId", "movieId", "rating"]).reset_index(drop=True)

# - Asegurar tipos finales
movies["movieId"]  = movies["movieId"].astype("int64")
ratings["userId"]  = ratings["userId"].astype("int64")
ratings["movieId"] = ratings["movieId"].astype("int64")
ratings["rating"]  = ratings["rating"].astype("float32")

# Convertir a categorías para ahorrar memoria
movies["title"] = movies["title"].astype("string")
if "genres" in movies.columns:
    movies["genres"] = movies["genres"].fillna("Unknown").astype("string")

# Resumen rápido
print("=== Resumen de datasets ===")
print(f"movies.shape  = {movies.shape}")
print(f"ratings.shape = {ratings.shape}")

n_users  = ratings["userId"].nunique()
n_items  = ratings["movieId"].nunique()
print(f"Usuarios únicos = {n_users}")
print(f"Películas únicas = {n_items}")

# Mostrar ejemplos
print("\nEjemplo movies:")
display(movies.head(5))
print("\nEjemplo ratings:")
display(ratings.head(5))

# Comprobar valores nulos restantes
print("\nNulos por columna (movies):")
print(movies.isna().sum())
print("\nNulos por columna (ratings):")
print(ratings.isna().sum())

=== Resumen de datasets ===
movies.shape  = (9742, 3)
ratings.shape = (100836, 4)
Usuarios únicos = 610
Películas únicas = 9724

Ejemplo movies:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Ejemplo ratings:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931



Nulos por columna (movies):
movieId    0
title      0
genres     0
dtype: int64

Nulos por columna (ratings):
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


## Matriz usuario–película

In [21]:
# Usar pivot_table de pandas -> dataframe denso (puede ser grande)
ratings_pivot = ratings.pivot_table(
    index="userId", columns="movieId", values="rating"
)

# Normalizar por usuario
user_means = ratings_pivot.mean(axis=1)
ratings_norm = ratings_pivot.sub(user_means, axis=0)

# Reemplazar NaN por 0 para crear matriz esparsa
ratings_filled = ratings_norm.fillna(0)

# Convertir a matriz esparsa CSR (más eficiente)
ratings_sparse = csr_matrix(ratings_filled.values)

print("Matriz usuario–película creada.")
print(f"Dimensiones: {ratings_sparse.shape}")
print(f"Usuarios: {ratings_sparse.shape[0]}, Películas: {ratings_sparse.shape[1]}")

# Guardar referencias: índices <-> IDs de películas
movie_ids = ratings_pivot.columns.to_list()   # lista de movieId en el orden de las columnas
user_ids  = ratings_pivot.index.to_list()     # lista de userId en el orden de las filas

print("\nEjemplo de movie_ids:", movie_ids[:5])
print("Ejemplo de user_ids:", user_ids[:5])

Matriz usuario–película creada.
Dimensiones: (610, 9724)
Usuarios: 610, Películas: 9724

Ejemplo de movie_ids: [1, 2, 3, 4, 5]
Ejemplo de user_ids: [1, 2, 3, 4, 5]


# Calcular similitud item–item (coseno)

In [41]:
# Calcular similitud coseno base entre películas
item_item_sim = cosine_similarity(ratings_sparse.T)

# Matriz binaria usuario–película para contar co-ocurrencias
binary_sparse = ratings_sparse.copy()
binary_sparse.data = np.ones_like(binary_sparse.data)

cooc = (binary_sparse.T @ binary_sparse).toarray().astype(np.float32)
print("Matriz de co-ocurrencias construida.")

# Shrinkage (suavizado)
alpha = 50.0  # mayor alpha = más penalización
sim_shrunk = (cooc / (cooc + alpha)) * item_item_sim
print(f"Similitud con shrinkage calculada (alpha={alpha}).")

# Mapas movieId <-> índice de columna
movieid_to_idx = {mid: i for i, mid in enumerate(movie_ids)}
idx_to_movieid = {i: mid for i, mid in enumerate(movie_ids)}

# Función para recomendar
def recomendar_similares_refinado(titulo, N=10, min_inter=5):
    fila = movies[movies["title"] == titulo]
    if fila.empty:
        print(f"Película '{titulo}' no encontrada.")
        return []
    mid = int(fila["movieId"].values[0])
    if mid not in movieid_to_idx:
        print(f"movieId {mid} no tiene suficientes datos.")
        return []
    
    idx = movieid_to_idx[mid]
    sims = sim_shrunk[idx].copy()

    # Filtrar pares con pocas co-ocurrencias
    pocos = cooc[idx] < float(min_inter)
    sims[pocos] = -1.0

    # Ordenar descendente
    orden = np.argsort(sims)[::-1]
    orden = [i for i in orden if i != idx and sims[i] > 0]
    top = orden[:N]

    recs = []
    for j in top:
        mid_sim = idx_to_movieid[j]
        titulo_sim = movies.loc[movies["movieId"] == mid_sim, "title"].values[0]
        recs.append((titulo_sim, float(sims[j]), int(cooc[idx, j])))
    return recs

# Prueba rápida
ejemplo = "Toy Story (1995)"
print(f"\nPelícula de consulta: {ejemplo} (refinado, min_inter=5, alpha={alpha})")
for t, s, n in recomendar_similares_refinado(ejemplo, N=10, min_inter=5):
    print(f"- {t} (sim={s:.4f}, co={n})")


Matriz de co-ocurrencias construida.
Similitud con shrinkage calculada (alpha=50.0).

Película de consulta: Toy Story (1995) (refinado, min_inter=5, alpha=50.0)
- Toy Story 2 (1999) (sim=0.2494, co=81)
- Aladdin (1992) (sim=0.2230, co=107)
- Back to the Future (1985) (sim=0.1881, co=106)
- Finding Nemo (2003) (sim=0.1671, co=87)
- Incredibles, The (2004) (sim=0.1659, co=76)
- Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) (sim=0.1615, co=107)
- Lion King, The (1994) (sim=0.1604, co=107)
- Toy Story 3 (2010) (sim=0.1514, co=43)
- Ghostbusters (a.k.a. Ghost Busters) (1984) (sim=0.1506, co=76)
- Star Wars: Episode IV - A New Hope (1977) (sim=0.1423, co=134)


## Ejemplo oficial de uso + observaciones

In [60]:
# Función para mostrar recomendaciones de forma clara
def mostrar_recomendaciones(titulo, N=10, min_inter=5):
    print(f"\n=== Recomendaciones para: '{titulo}' ===")
    recs = recomendar_similares_refinado(titulo, N=N, min_inter=min_inter)
    if not recs:
        print("No se encontraron recomendaciones.")
        return
    for t, s, n in recs:
        print(f"- {t} (sim={s:.4f}, co={n})")

# Ejemplo oficial
ejemplo = "Toy Story (1995)"
mostrar_recomendaciones(ejemplo, N=10, min_inter=5)

# Otro ejemplo
ejemplo2 = "Pulp Fiction (1994)"
mostrar_recomendaciones(ejemplo2, N=10, min_inter=5)


=== Recomendaciones para: 'Toy Story (1995)' ===
- Toy Story 2 (1999) (sim=0.2494, co=81)
- Aladdin (1992) (sim=0.2230, co=107)
- Back to the Future (1985) (sim=0.1881, co=106)
- Finding Nemo (2003) (sim=0.1671, co=87)
- Incredibles, The (2004) (sim=0.1659, co=76)
- Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) (sim=0.1615, co=107)
- Lion King, The (1994) (sim=0.1604, co=107)
- Toy Story 3 (2010) (sim=0.1514, co=43)
- Ghostbusters (a.k.a. Ghost Busters) (1984) (sim=0.1506, co=76)
- Star Wars: Episode IV - A New Hope (1977) (sim=0.1423, co=134)

=== Recomendaciones para: 'Pulp Fiction (1994)' ===
- Usual Suspects, The (1995) (sim=0.3501, co=161)
- Fight Club (1999) (sim=0.3466, co=158)
- Silence of the Lambs, The (1991) (sim=0.3134, co=205)
- Shawshank Redemption, The (1994) (sim=0.2931, co=221)
- Godfather, The (1972) (sim=0.2860, co=131)
- Reservoir Dogs (1992) (sim=0.2823, co=115)
- Seven (a.k.a. Se7en) (1995) (sim=0.2669, co=173)
- Goodfellas (1990)

## Conclusión

En este trabajo practiqué la construcción de un sistema de recomendación item–item usando el conjunto MovieLens (ml-latest-small). La idea fue directa: encontrar películas parecidas observando el comportamiento de los usuarios, sin utilizar metadatos como el género. Empecé cargando movies.csv y ratings.csv, verificando columnas y valores ausentes. A continuación, construí la matriz usuario × película (610 usuarios y 9.724 películas) y la normalicé por usuario (restando la media individual) para reducir el sesgo de quienes dan notas siempre altas o bajas. Convertí la matriz al formato disperso para ahorrar memoria y acelerar los cálculos.

Con la matriz lista, calculé la similitud del coseno entre las columnas (cada columna representa una película por su vector de ratings normalizados). Para evitar falsos positivos cuando pocos usuarios evaluaron el mismo par de películas, afiné la medida de similitud con dos ideas: (1) contar las coocurrencias, que es el número de usuarios que evaluaron ambas películas (co), y (2) aplicar un shrinkage controlado por un parámetro alpha (utilicé 50). La fórmula quedó como (co / (co + alpha)) * sim_coseno, lo que reduce la similitud cuando hay poca superposición de usuarios y la mantiene cuando hay suficiente evidencia.

Implementé la función recomendar_similares_refinado(título, N, min_inter) para devolver las N películas más cercanas, mostrando también la similitud ajustada (sim) y el número de co-usuarios (co). En las pruebas, los resultados fueron coherentes: para Toy Story (1995) aparecieron Toy Story 2 y Toy Story 3; para Pulp Fiction (1994) surgieron The Usual Suspects y Fight Club. Estas listas tienen sentido porque muchos usuarios que disfrutan de una también valoran bien las otras.

Como limitaciones, observé el problema clásico de cold start (películas nuevas o con pocas valoraciones) y la dispersión de la matriz, que reduce la cobertura. Aun así, el método es simple, transparente, reproducible y cumple el objetivo del ejercicio: mostrar en la práctica cómo los patrones de evaluación de los usuarios permiten recomendar películas similares de forma explicable.